# Assignment 21: LangChain Document Loaders & Text Splitters
**Student Name:** Parth Dadhaniya  
**Course:** GenAI Course Assignments  
**Role:** Generative AI Engineer  
**Scenario:** Personal Knowledge Assistant (Ingestion & Chunking Foundation)

---

### Overview
Before building RAG or chatbots, raw text data from different sources (text notes, CSVs, PDFs, websites) must be loaded and broken into small chunks so LLMs can process them without exceeding context limits or losing context.


## Part 1: Document Loaders in LangChain

### Setup Dataset
We create a `data/` folder containing:
- `notes.txt` (company notes)
- `data.csv` (tabular employee data)
- `sample.pdf` (multi-page PDF)
- `handbook.md` (structured markdown)


In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

os.makedirs("data", exist_ok=True)

# 1. notes.txt
notes_content = '''GenAI Knowledge Assistant - System Architecture & Overview

Introduction:
The Personal Knowledge Assistant is designed to help team members quickly query internal company documentation, policies, and project notes. It eliminates the friction of manual search across scattered drives and repositories.

Data Ingestion Layer:
The ingestion subsystem handles heterogeneous data sources including text notes, tabular spreadsheets, customer reports, and external web documentation. LangChain document loaders standardize these formats into unified Document objects consisting of page_content and metadata.

Text Splitting & Chunking Strategy:
Large unstructured documents cannot be fed into LLM prompts directly due to context window constraints and retrieval noise. Chunking breaks large documents into smaller, semantically coherent passages. We evaluate length-based splitters, recursive character splitters, and structure-aware splitters to find the optimal chunk size and overlap for accurate semantic retrieval.

Compliance and Security:
All processed documents must follow strict enterprise data governance. Personally identifiable information (PII) must be masked before entering embedding pipelines. Proprietary source codes and internal financial projections must remain on local premises.
'''

with open("data/notes.txt", "w", encoding="utf-8") as f:
    f.write(notes_content.strip())

# 2. data.csv
csv_content = '''id,name,department,role,project,experience_years
101,Aarav Sharma,Engineering,Backend Engineer,GenAI Search Platform,4
102,Diya Patel,Data Science,NLP Specialist,Customer Feedback Classifier,3
103,Rohan Mehta,DevOps,Cloud Architect,Internal Knowledge Infrastructure,6
104,Priya Nair,Product,Product Manager,Enterprise AI Assistant,5
105,Vikram Singh,Security,Compliance Officer,Data Privacy and Governance,8
'''

with open("data/data.csv", "w", encoding="utf-8") as f:
    f.write(csv_content.strip())

# 3. handbook.md
md_content = '''# Enterprise AI Assistant Handbook

## 1. Overview
The Enterprise AI Assistant is built to index company knowledge and provide verifiable answers with source citations.

## 2. Document Ingestion Rules
### 2.1 File Formats
We support plain text files, CSV tables, PDF reports, and web documentation.

### 2.2 Metadata Standards
Every loaded document must retain metadata including source file path, author, and timestamp.

## 3. Chunking Guidelines
### 3.1 Chunk Size
Standard chunk size should be between 300 and 500 characters for fine-grained retrieval.

### 3.2 Chunk Overlap
A 10% to 20% overlap prevents loss of context at chunk boundaries.
'''

with open("data/handbook.md", "w", encoding="utf-8") as f:
    f.write(md_content.strip())

# 4. sample.pdf using reportlab
pdf_path = "data/sample.pdf"
c = canvas.Canvas(pdf_path, pagesize=letter)

c.setFont("Helvetica-Bold", 18)
c.drawString(72, 720, "Company AI Knowledge Base - Page 1")
c.setFont("Helvetica", 12)
c.drawString(72, 680, "Section 1: Introduction to Generative AI Systems")
text_p1 = [
    "Generative AI models excel at synthesizing information, generating responses,",
    "and retrieving insights across massive collections of unstructured text data.",
    "However, large foundation models have limited context windows and cannot access",
    "private enterprise documents without explicit Retrieval-Augmented Generation (RAG).",
    "The first phase of any RAG pipeline is robust document ingestion and chunking."
]
y = 650
for line in text_p1:
    c.drawString(72, y, line)
    y -= 20

c.showPage()

c.setFont("Helvetica-Bold", 18)
c.drawString(72, 720, "Company AI Knowledge Base - Page 2")
c.setFont("Helvetica", 12)
c.drawString(72, 680, "Section 2: Chunking and Embedding Considerations")
text_p2 = [
    "When splitting large PDF documents, retaining page numbers and headings in",
    "metadata is critical for accurate source citations.",
    "Recursive splitting ensures that paragraphs and sentences remain intact rather",
    "than being sliced mid-sentence, which preserves semantic coherence.",
    "This completes the reference sample PDF document for testing LangChain loaders."
]
y = 650
for line in text_p2:
    c.drawString(72, y, line)
    y -= 20

c.save()
print("Created files:", os.listdir("data"))


### Task 1: Text Loader
Loading `notes.txt` using `TextLoader`.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import TextLoader

# loading notes.txt
loader = TextLoader("data/notes.txt", encoding="utf-8")
docs = loader.load()

print("Number of documents loaded:", len(docs))
print("
Metadata:")
print(docs[0].metadata)

print("
Page Content Preview:")
print(docs[0].page_content[:300])
print(f"
Total characters: {len(docs[0].page_content)}")


### Task 2: CSV Loader
Loading tabular data with `CSVLoader`. Each row becomes a separate Document.


In [ ]:
from langchain_community.document_loaders import CSVLoader

# loading csv file where each row is converted to a document
loader = CSVLoader("data/data.csv", encoding="utf-8")
csv_docs = loader.load()

print("Total documents loaded from CSV:", len(csv_docs))

print("
Sample Document 1:")
print(csv_docs[0].page_content)
print("Metadata:", csv_docs[0].metadata)

print("
Sample Document 2:")
print(csv_docs[1].page_content)
print("Metadata:", csv_docs[1].metadata)


### Task 3: PDF Loader
Loading `sample.pdf` using `PyPDFLoader` to extract page text and metadata.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# loading pdf document
loader = PyPDFLoader("data/sample.pdf")
pages = loader.load()

print("Total pages in PDF:", len(pages))

print("
--- Page 1 Content ---")
print(pages[0].page_content)
print("Metadata:", pages[0].metadata)

if len(pages) > 1:
    print("
--- Page 2 Content ---")
    print(pages[1].page_content)
    print("Metadata:", pages[1].metadata)


### Task 4: Directory Loader
Loading all files inside `data/` using `DirectoryLoader` with proper loaders for each format.


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, CSVLoader, PyPDFLoader

# loading all files inside data/ using DirectoryLoader
txt_loader = DirectoryLoader("data", glob="**/*.txt", loader_cls=TextLoader)
csv_loader = DirectoryLoader("data", glob="**/*.csv", loader_cls=CSVLoader)
pdf_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyPDFLoader)

all_docs = []
for name, ldr in [("Text", txt_loader), ("CSV", csv_loader), ("PDF", pdf_loader)]:
    d = ldr.load()
    all_docs.extend(d)
    print(f"Loaded {len(d)} documents from {name} files")

print(f"
Total documents loaded from directory: {len(all_docs)}")

print("
Loaded documents summary:")
for i, d in enumerate(all_docs, 1):
    source = d.metadata.get("source", "unknown")
    page = d.metadata.get("page", "-")
    print(f"{i}. source: {source} | page: {page} | length: {len(d.page_content)} chars")


### Task 5: WebBase Loader
Extracting text from a public web page (Wikipedia) using `WebBaseLoader`.


In [ ]:
import os
from langchain_community.document_loaders import WebBaseLoader

os.environ["USER_AGENT"] = "PersonalKnowledgeAssistant/1.0"

# loading webpage from wikipedia
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
print(f"Loading webpage: {url}")

loader = WebBaseLoader(url)
web_docs = loader.load()

print("Documents loaded:", len(web_docs))
print("Page Title:", web_docs[0].metadata.get("title"))
print("Source URL:", web_docs[0].metadata.get("source"))

# clean whitespace for preview
text = " ".join(web_docs[0].page_content.split())
print("
First 500 characters:")
print(text[:500])


## Part 2: Text Splitters in LangChain

### Task 6: Why Text Splitting is Required (Conceptual)

#### 1. Why large documents cannot be directly passed to LLMs?
1. **Context Window Limits:** Models have token limits (8k, 32k tokens). Full documents or books simply won't fit inside a single prompt.
2. **Lost in the Middle:** When given too much context, models get confused and miss specific facts buried in the middle of long texts.
3. **High Cost & Latency:** Passing entire documents on every user query is slow and wastes API tokens.

#### 2. What problems does chunking solve in GenAI systems?
1. **Precise Retrieval:** Vector databases search and return only the 2-3 specific chunks needed to answer the question.
2. **Less Noise:** The model receives only relevant passages, reducing hallucinations.
3. **Better Embeddings:** Focused 300-500 character chunks capture a single clear idea, making vector embeddings much more accurate.


### Task 7: Length-Based Text Splitter
Testing `CharacterTextSplitter` with `chunk_size=250` and `chunk_overlap=40`.


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# load document
docs = TextLoader("data/notes.txt", encoding="utf-8").load()
print(f"Original text length: {len(docs[0].page_content)} characters")

chunk_size = 250
chunk_overlap = 40

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

chunks = splitter.split_documents(docs)

print(f"
Settings: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
print(f"Total chunks created: {len(chunks)}")

print("
--- Sample Chunk 1 ---")
print(f"Length: {len(chunks[0].page_content)} chars")
print(chunks[0].page_content)

print("
--- Sample Chunk 2 ---")
print(f"Length: {len(chunks[1].page_content)} chars")
print(chunks[1].page_content)


### Task 8: Text Structure-Based Splitter
Testing `RecursiveCharacterTextSplitter` and comparing against `CharacterTextSplitter`.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# recursive character splitter
rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", " ", ""]
)
rec_chunks = rec_splitter.split_documents(docs)

print("Comparison: CharacterTextSplitter vs RecursiveCharacterTextSplitter")
print(f"- Character splitter chunks: {len(chunks)}")
print(f"- Recursive splitter chunks: {len(rec_chunks)}")

print("
--- Recursive Chunk 1 ---")
print(f"Length: {len(rec_chunks[0].page_content)} chars")
print(rec_chunks[0].page_content)

print("
--- Recursive Chunk 2 ---")
print(f"Length: {len(rec_chunks[1].page_content)} chars")
print(rec_chunks[1].page_content)


### Task 9: Document Structure-Based Splitting
Splitting markdown files by headings using `MarkdownHeaderTextSplitter` and preserving PDF page numbers.


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

with open("data/handbook.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()

# split based on markdown headers
headers_to_split = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split)
md_chunks = md_splitter.split_text(markdown_text)

print(f"Total markdown chunks: {len(md_chunks)}")
for i, c in enumerate(md_chunks[:3], 1):
    print(f"
Chunk {i}:")
    print("Metadata:", c.metadata)
    print("Content:", c.page_content.strip())


### Task 10: Semantic Meaning-Based Splitting (Conceptual + Demo)
Instead of splitting text after a fixed number of characters, semantic chunking groups sentences together that talk about the same idea, creating a new chunk only when the topic changes.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    # Topic 1: Machine Learning
    "Machine learning models learn patterns from training data.",
    "Deep neural network models optimize internal weights during training.",
    "These machine learning models evaluate performance on validation sets.",
    # Topic 2: Databases
    "Relational database tables store structured business records safely.",
    "SQL queries retrieve structured data from database tables efficiently.",
    # Topic 3: Cloud & Containers
    "Docker containers package applications into portable deployable units.",
    "Cloud Kubernetes clusters orchestrate and scale application containers."
]

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(sentences)

threshold = 0.10
chunks = []
current_chunk = [sentences[0]]

for i in range(len(sentences) - 1):
    sim = cosine_similarity(vectors[i], vectors[i + 1])[0][0]
    print(f"Similarity between sentence {i+1} and {i+2}: {sim:.3f}")
    if sim < threshold:
        # topic changed, start a new chunk
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i + 1]]
    else:
        # same topic, keep together
        current_chunk.append(sentences[i + 1])
chunks.append(" ".join(current_chunk))

print(f"
Formed {len(chunks)} semantic chunks based on similarity threshold ({threshold}):")
for idx, c in enumerate(chunks, 1):
    print(f"
Chunk {idx}:")
    print(f"'{c}'")


## Part 3: Mini Integration Task

### Task 11: Unified Preprocessing Pipeline
We create a function `load_and_split_documents(path_or_url)` that accepts a folder or web URL, loads documents, and splits them into chunks.


In [ ]:
# function to load and split documents from either a local path or web url
def load_and_split_documents(path_or_url, chunk_size=400, chunk_overlap=50):
    raw_docs = []
    
    # check if web url
    if str(path_or_url).startswith(("http://", "https://")):
        print(f"Loading from URL: {path_or_url}")
        loader = WebBaseLoader(path_or_url)
        raw_docs = loader.load()
        
    # check if directory
    elif os.path.isdir(path_or_url):
        print(f"Loading files from folder: {path_or_url}")
        for root, _, files in os.walk(path_or_url):
            for file in files:
                filepath = os.path.join(root, file)
                ext = os.path.splitext(file)[1].lower()
                if ext == ".txt":
                    raw_docs.extend(TextLoader(filepath, encoding="utf-8").load())
                elif ext == ".csv":
                    raw_docs.extend(CSVLoader(filepath, encoding="utf-8").load())
                elif ext == ".pdf":
                    raw_docs.extend(PyPDFLoader(filepath).load())

    # check if single file
    elif os.path.isfile(path_or_url):
        print(f"Loading single file: {path_or_url}")
        ext = os.path.splitext(path_or_url)[1].lower()
        if ext == ".txt":
            raw_docs = TextLoader(path_or_url, encoding="utf-8").load()
        elif ext == ".csv":
            raw_docs = CSVLoader(path_or_url, encoding="utf-8").load()
        elif ext == ".pdf":
            raw_docs = PyPDFLoader(path_or_url).load()
    else:
        raise ValueError(f"Invalid path or url: {path_or_url}")

    print(f"Loaded {len(raw_docs)} raw documents")

    # split documents into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(raw_docs)
    print(f"Created {len(chunks)} chunks")
    return chunks

print("Testing on local data folder:")
local_chunks = load_and_split_documents("data", chunk_size=350, chunk_overlap=40)
print("Sample chunk metadata:", local_chunks[0].metadata)

print("
" + "-" * 50)
print("Testing on web URL:")
web_chunks = load_and_split_documents("https://en.wikipedia.org/wiki/Artificial_intelligence", chunk_size=400, chunk_overlap=50)
print("Sample web chunk metadata:", web_chunks[0].metadata)


### Task 12: Observations & Insights

#### 1. Which loader is used for which data type?
- `.txt` files: `TextLoader` (loads full text file as a single document)
- `.csv` files: `CSVLoader` (loads each row as a separate document with columns as key-values)
- `.pdf` files: `PyPDFLoader` (loads page by page and stores page numbers in metadata)
- folders: `DirectoryLoader` (scans folder and uses loaders matching file extensions)
- webpages: `WebBaseLoader` (scrapes html text using BeautifulSoup from public urls)

#### 2. Best splitter for each data type:
- **Small text:** `CharacterTextSplitter` (simple splitting is enough since text is already short)
- **Large PDFs:** `PyPDFLoader` + `RecursiveCharacterTextSplitter` (keeps page numbers in metadata and respects paragraph boundaries so tables and sentences don't get cut in half)
- **Web data:** `RecursiveCharacterTextSplitter` or Markdown/HTML header splitters (preserves header hierarchy and keeps article sections separate from navigation bars)

#### 3. Why chunk overlap is important?
- **Context Preservation:** Overlap prevents sentences from being cut right at the boundary.
- **Better Search Results:** If an answer spans across two paragraphs, overlap makes sure at least one chunk contains the full context.
- **Reduces Hallucinations:** The LLM gets complete sentences instead of cut-off fragments.
